# 🛡️ FRONTIER-OS // C4ISR Master GPU Training Suite (Google Colab)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/00_FRONTIER_C4ISR_Master_Colab_Training_Suite.ipynb)

Este notebook executa o pipeline completo de treinamento em **GPU (CUDA Mixed Precision FP16)** para todos os **5 Subsistemas de Deep Learning** do **FRONTIER-OS**:
1. **`01-maritime-coastline` (USCG / US Navy):** `TrajectoryAnomalyLSTM` -> Exporta `maritime_lstm.onnx`
2. **`02-airspace-shield` (USAF / CBP AMO):** `FlightStateAutoencoder` -> Exporta `airspace_ae.onnx`
3. **`03-narcotics-intel` (DEA / CBP / HIDTA):** `SpatioTemporalGNN` -> Exporta `narcotics_stgnn.onnx`
4. **`04-criminal-interstate` (FBI / State Police):** `ArcFace-512 Biometrics & Qdrant Embeddings`
5. **`05-geospatial-recon` (US Army / NGA):** `YOLOv8-Military Satellite Detector` -> Exporta `frontier_sat_yolov8.onnx`


In [ ]:
# 1. Verificar GPU (NVIDIA T4 / L4 / A100) e clonar o repositório oficial
!nvidia-smi
import os, sys

if not os.path.exists("FRONTIER-OS-Frontier-Operating-System-"):
    !git clone https://github.com/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-.git

%cd FRONTIER-OS-Frontier-Operating-System-
!pip install -q torch torchvision torchaudio onnx onnxscript onnxruntime-gpu httpx tqdm geopandas shapely pyarrow matplotlib
sys.path.insert(0, os.getcwd())
print("✅ Ambiente GPU configurado com sucesso!")


In [ ]:
# 2. Baixar Dados Reais das Agências do Governo dos EUA (Census TIGER, FBI Wanted, OpenSky, CBP, MarineCadastre)
!python3 infra/scripts/download_real_gov_datasets.py


In [ ]:
# 3. Treinar Todos os 4 Modelos de Deep Learning em GPU e Exportar para ONNX (Dynamic Axes)
import time
t0 = time.time()

print("🚢 [1/4] Treinando Subsistema 01: Maritime AIS LSTM (USCG / US Navy)...")
from ml_training.maritime_lstm.train_ais_lstm import train_maritime_lstm_pipeline
train_maritime_lstm_pipeline(epochs=20, batch_size=32)

print("\n✈️ [2/4] Treinando Subsistema 02: Airspace ADS-B Autoencoder (USAF / FAA)...")
from ml_training.airspace_autoencoder.train_adsb_ae import train_airspace_ae_pipeline
train_airspace_ae_pipeline(epochs=25, batch_size=32)

print("\n⚠️ [3/4] Treinando Subsistema 03: Narcotics Spatio-Temporal GNN (DEA / CBP)...")
from ml_training.narcotics_gnn.train_stgnn import train_stgnn_pipeline
train_stgnn_pipeline(epochs=30)

print("\n🛰️ [4/4] Treinando Subsistema 05: Satellite Recon YOLOv8 (US Army / NGA)...")
from ml_training.satellite_vision.train_yolo_military import train_satellite_yolo_pipeline
train_satellite_yolo_pipeline()

print(f"\n✅ Todos os modelos treinados e exportados para ONNX em {time.time() - t0:.2f}s!")
!ls -lh ml-training/weights/


In [ ]:
# 4. Validar Inferência ONNX Runtime e Empacotar Pesos (.zip) para Download ou Deploy OCI
import zipfile
from pathlib import Path
import onnxruntime as ort
import numpy as np

weights_dir = Path("ml-training/weights")
for onnx_file in weights_dir.glob("*.onnx"):
    sess = ort.InferenceSession(str(onnx_file), providers=["CPUExecutionProvider"])
    inputs = {inp.name: f"{inp.shape} ({inp.type})" for inp in sess.get_inputs()}
    print(f"✔️ ONNX Validado: {onnx_file.name} | Inputs: {inputs}")

zip_path = "frontier_c4isr_onnx_weights.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in weights_dir.iterdir():
        if f.is_file():
            zf.write(f, arcname=f.name)

print(f"\n📦 Arquivo pronto para download: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Execute no Google Colab para iniciar o download automático pelo navegador.")
